# From-Scratch TTS for Kaggle (LJSpeech, single-speaker, end-to-end)
This notebook is designed for a **graduation-project-safe submission path**:

- **from scratch** acoustic model (random initialization, no pretrained checkpoints)
- **single-speaker** training on **LJSpeech**
- **phoneme input (with safe character fallback)**
- **location-sensitive attention**
- **guided attention loss**
- **stop token**
- **postnet**
- **AMP mixed precision**
- **checkpoint resume**
- **text -> audio** inference at the end

## What this notebook guarantees
If training runs successfully, it gives you a complete path:

`text -> tokens -> mel-spectrogram -> waveform (.wav)`

The waveform generation uses **Griffin-Lim** as a reliable, fully from-scratch baseline.  
That is intentional: it is the safest way to guarantee an end-to-end submission on Kaggle.

## Kaggle setup
Attach a dataset containing **LJSpeech-1.1**.  
The notebook auto-detects a folder that contains:

- `metadata.csv`
- `wavs/`

Examples:
- `/kaggle/input/ljspeech11/LJSpeech-1.1`
- `/kaggle/input/datasets/rahulbhalley/ljspeech11/LJSpeech-1.1`


In [ ]:
# =========================
# 1) Install requirements
# =========================
!pip -q install phonemizer unidecode soundfile
!apt-get -qq update
!apt-get -qq install -y espeak-ng libespeak-ng1 libespeak-ng-dev > /dev/null
print("Dependencies installed.")


In [ ]:
# =========================
# 2) Imports
# =========================
import os
import re
import math
import json
import time
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from unidecode import unidecode

PHONEMIZER_OK = False
try:
    espeak_candidates = sorted(
        glob.glob("/usr/lib/x86_64-linux-gnu/libespeak-ng.so*") +
        glob.glob("/lib/x86_64-linux-gnu/libespeak-ng.so*")
    )
    if len(espeak_candidates) > 0:
        os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = espeak_candidates[-1]

    from phonemizer import phonemize
    from phonemizer.backend.espeak.wrapper import EspeakWrapper

    if "PHONEMIZER_ESPEAK_LIBRARY" in os.environ:
        EspeakWrapper.set_library(os.environ["PHONEMIZER_ESPEAK_LIBRARY"])

    PHONEMIZER_OK = True
except Exception as e:
    print("Phonemizer setup failed. Notebook will fall back to character tokens.")
    print("Reason:", repr(e))

warnings.filterwarnings("ignore")

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("PHONEMIZER_OK:", PHONEMIZER_OK)


In [ ]:
# =========================
# 3) Configuration
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Audio
SAMPLE_RATE = 22050
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 256
N_MELS = 80
FMIN = 0
FMAX = 8000
MAX_WAV_VALUE = 32768.0

# Data filtering
MIN_TEXT_LEN = 1
MAX_TEXT_LEN = 220
MIN_MEL_LEN = 20
MAX_MEL_LEN = 900

# Splits
VAL_SIZE = 300

# Loader
BATCH_SIZE = 24
NUM_WORKERS = 2
PIN_MEMORY = True

# Acoustic model
EMBED_DIM = 256
ENC_HIDDEN = 256
DEC_HIDDEN = 512
ATTN_DIM = 128
LOCATION_FILTERS = 32
LOCATION_KERNEL = 31
PRENET_DIM = 256
POSTNET_CHANNELS = 512
POSTNET_KERNEL = 5
REDUCTION_FACTOR = 2

# Training
EPOCHS = 20
LR = 2e-4
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 5
USE_AMP = torch.cuda.is_available()

# Teacher forcing schedule
TEACHER_FORCING_START = 1.0
TEACHER_FORCING_END = 0.75

# Guided attention schedule
GUIDED_ATTN_MAX_LAMBDA = 0.15
GUIDED_ATTN_MIN_LAMBDA = 0.02
GUIDED_ATTN_G = 0.2

# Stop token
STOP_POS_WEIGHT = 8.0

# Inference
DEFAULT_STOP_THRESHOLD = 0.45
DEFAULT_MIN_DECODER_STEPS = 10
DEFAULT_MAX_DECODER_STEPS = 400

WORK_DIR = Path("/kaggle/working/final_tts_from_scratch")
CKPT_DIR = WORK_DIR / "checkpoints"
SAMPLE_DIR = WORK_DIR / "samples"
PLOT_DIR = WORK_DIR / "plots"

for p in [WORK_DIR, CKPT_DIR, SAMPLE_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Work dir:", WORK_DIR)


In [ ]:
# =========================
# 4) Dataset auto-detection
# =========================
def find_ljspeech_root(search_root="/kaggle/input"):
    candidates = []
    for root, dirs, files in os.walk(search_root):
        if "metadata.csv" in files and "wavs" in dirs:
            candidates.append(root)
    return candidates

candidates = find_ljspeech_root("/kaggle/input")
if len(candidates) == 0:
    raise FileNotFoundError(
        "Could not find LJSpeech. Please attach a dataset that contains metadata.csv and wavs/."
    )

print("Found dataset candidates:")
for c in candidates:
    print(" -", c)

LJSPEECH_PATH = Path(candidates[0])
CSV_PATH = LJSPEECH_PATH / "metadata.csv"
WAVS_DIR = LJSPEECH_PATH / "wavs"

print("\nUsing:", LJSPEECH_PATH)
assert CSV_PATH.exists() and WAVS_DIR.exists()


In [ ]:
# =========================
# 5) Text cleaning + tokenization helpers
# =========================
_whitespace_re = re.compile(r"\s+")
_allowed_text = re.compile(r"[^a-zA-Z0-9 ,.!?;:'\"()\-\[\]]+")

def clean_text(text: str) -> str:
    text = str(text)
    text = unidecode(text)
    text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    text = _allowed_text.sub(" ", text)
    text = _whitespace_re.sub(" ", text).strip()
    return text

def text_to_token_string(text: str) -> str:
    text = clean_text(text)

    if PHONEMIZER_OK:
        try:
            ph = phonemize(
                [text],
                language="en-us",
                backend="espeak",
                strip=True,
                preserve_punctuation=True,
                with_stress=True
            )[0]
            ph = _whitespace_re.sub(" ", ph).strip()
            if len(ph) > 0:
                return ph
        except Exception as e:
            print("Phonemizer failed at runtime, switching to character fallback.")
            print("Reason:", repr(e))

    return " ".join(list(text))

print(text_to_token_string("Hello! This is my graduation project."))


In [ ]:
# =========================
# 6) Load metadata, filter, split, build vocab
#    (ultra-safe version without pandas.read_csv)
# =========================
def read_ljspeech_metadata(csv_path):
    rows = []
    with open(csv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            parts = line.split("|")
            if len(parts) < 2:
                continue
            utt_id = parts[0]
            raw_text = parts[1] if len(parts) > 1 else ""
            norm_text = parts[2] if len(parts) > 2 else raw_text
            rows.append((utt_id, raw_text, norm_text))
    return rows

raw_rows = read_ljspeech_metadata(CSV_PATH)
print("Metadata rows read:", len(raw_rows))

records = []
for utt_id, raw_text, norm_text in raw_rows:
    wav_path = str(WAVS_DIR / f"{utt_id}.wav")
    if not os.path.exists(wav_path):
        continue

    text = clean_text(norm_text if norm_text else raw_text)
    if not (MIN_TEXT_LEN <= len(text) <= MAX_TEXT_LEN):
        continue

    try:
        tok_str = text_to_token_string(text)
        if len(tok_str) == 0:
            continue
    except Exception:
        continue

    records.append({
        "id": utt_id,
        "raw_text": raw_text,
        "norm_text": norm_text,
        "text": text,
        "wav_path": wav_path,
        "tokens": tok_str,
    })

meta = pd.DataFrame.from_records(records)
del raw_rows, records

print("Rows after safe tokenization/filtering:", len(meta))
meta.head()


In [ ]:
# =========================
# 7) Build vocabulary
# =========================
special_tokens = ["<pad>", "<sos>", "<eos>"]
token_set = set()

for tok_str in meta["tokens"].tolist():
    for tok in tok_str.split():
        token_set.add(tok)

vocab = special_tokens + sorted(list(token_set))
stoi = {s: i for i, s in enumerate(vocab)}
itos = {i: s for s, i in stoi.items()}

PAD_ID = stoi["<pad>"]
SOS_ID = stoi["<sos>"]
EOS_ID = stoi["<eos>"]

print("Vocab size:", len(vocab))
print("Sample tokens:", vocab[:20])


In [ ]:
# =========================
# 8) Audio + mel helpers
# =========================
def load_wav(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    wav = np.clip(wav, -1.0, 1.0)
    return wav

def wav_to_log_mel(wav):
    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=1.0
    )
    mel = np.log(np.maximum(mel, 1e-5)).astype(np.float32)
    return mel.T  # [T, n_mels]

def log_mel_to_audio_griffinlim(mel_log, n_iter=80):
    mel = np.exp(mel_log.T)  # [n_mels, T]
    audio = librosa.feature.inverse.mel_to_audio(
        mel,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        power=1.0,
        n_iter=n_iter
    )
    audio = np.clip(audio, -1.0, 1.0)
    return audio.astype(np.float32)

def approx_mel_len_from_wav_path(path):
    info = sf.info(path)
    frames = info.frames
    if info.samplerate != SAMPLE_RATE:
        frames = int(frames * SAMPLE_RATE / info.samplerate)
    return math.ceil(frames / HOP_LENGTH)


In [ ]:
# =========================
# 9) Final filtering by audio length
#    (memory-safe version)
# =========================
filtered_records = []

for _, row in meta.iterrows():
    try:
        ml = approx_mel_len_from_wav_path(row["wav_path"])
        if MIN_MEL_LEN <= ml <= MAX_MEL_LEN:
            item = row.to_dict()
            item["mel_len"] = ml
            filtered_records.append(item)
    except Exception:
        pass

meta = pd.DataFrame.from_records(filtered_records)
del filtered_records

print("Rows after audio-length filtering:", len(meta))
print(meta["mel_len"].describe())


In [ ]:
# =========================
# 10) Train/val split
# =========================
meta = meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_size = min(VAL_SIZE, max(100, len(meta) // 20))
val_df = meta.iloc[:val_size].reset_index(drop=True)
train_df = meta.iloc[val_size:].reset_index(drop=True)

print("Train:", len(train_df), "| Val:", len(val_df))


In [ ]:
# =========================
# 11) Dataset + collate
# =========================
class LJTTSDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def token_string_to_ids(self, token_str):
        toks = token_str.split()
        ids = [SOS_ID] + [stoi[t] for t in toks if t in stoi] + [EOS_ID]
        return np.array(ids, dtype=np.int64)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids = self.token_string_to_ids(row["tokens"])

        wav = load_wav(row["wav_path"])
        mel = wav_to_log_mel(wav)

        return {
            "ids": ids,
            "mel": mel,
            "text": row["text"],
            "tokens": row["tokens"],
        }

def collate_batch(batch):
    batch = sorted(batch, key=lambda x: len(x["ids"]), reverse=True)

    input_lens = [len(x["ids"]) for x in batch]
    mel_lens = [x["mel"].shape[0] for x in batch]

    max_input_len = max(input_lens)
    max_mel_len = max(mel_lens)

    if max_mel_len % REDUCTION_FACTOR != 0:
        max_mel_len += REDUCTION_FACTOR - (max_mel_len % REDUCTION_FACTOR)

    B = len(batch)

    ids = torch.full((B, max_input_len), PAD_ID, dtype=torch.long)
    mels = torch.zeros((B, max_mel_len, N_MELS), dtype=torch.float32)
    stop_targets = torch.zeros((B, max_mel_len // REDUCTION_FACTOR), dtype=torch.float32)

    texts = []
    tokens = []

    for i, item in enumerate(batch):
        cur_ids = torch.tensor(item["ids"], dtype=torch.long)
        cur_mel = torch.tensor(item["mel"], dtype=torch.float32)

        ids[i, :cur_ids.size(0)] = cur_ids
        mels[i, :cur_mel.size(0)] = cur_mel

        reduced_len = math.ceil(cur_mel.size(0) / REDUCTION_FACTOR)
        stop_targets[i, reduced_len - 1 :] = 1.0

        texts.append(item["text"])
        tokens.append(item["tokens"])

    return (
        ids,
        torch.tensor(input_lens, dtype=torch.long),
        mels,
        torch.tensor(mel_lens, dtype=torch.long),
        stop_targets,
        texts,
        tokens,
    )

train_loader = DataLoader(
    LJTTSDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
    drop_last=False,
)

val_loader = DataLoader(
    LJTTSDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
    drop_last=False,
)

next(iter(train_loader))[0].shape


In [ ]:
# =========================
# 12) Model components
# =========================
class Prenet(nn.Module):
    def __init__(self, in_dim, sizes=(256, 256), dropout=0.5):
        super().__init__()
        layers = []
        last = in_dim
        for s in sizes:
            layers += [nn.Linear(last, s), nn.ReLU()]
            last = s
        self.layers = nn.ModuleList([nn.Linear(in_dim if i == 0 else sizes[i-1], sizes[i]) for i in range(len(sizes))])
        self.dropout = dropout

    def forward(self, x):
        # Keep dropout active in both train and inference like Tacotron-style prenet
        for linear in self.layers:
            x = F.relu(linear(x))
            x = F.dropout(x, p=self.dropout, training=True)
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, embed_dim, kernel_size=5, padding=2),
                nn.BatchNorm1d(embed_dim),
                nn.ReLU(),
                nn.Dropout(0.5)
            ) for _ in range(3)
        ])
        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x, lengths):
        x = self.embedding(x)          # [B, T, E]
        x = x.transpose(1, 2)          # [B, E, T]
        for conv in self.convs:
            x = conv(x)
        x = x.transpose(1, 2)          # [B, T, E]

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        packed_out, _ = self.bilstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        return out                     # [B, T, 2H]

class LocationLayer(nn.Module):
    def __init__(self, attn_n_filters, attn_kernel_size, attn_dim):
        super().__init__()
        padding = (attn_kernel_size - 1) // 2
        self.location_conv = nn.Conv1d(
            2, attn_n_filters, kernel_size=attn_kernel_size, padding=padding, bias=False
        )
        self.location_dense = nn.Linear(attn_n_filters, attn_dim, bias=False)

    def forward(self, attn_cat):
        # attn_cat: [B, 2, T_enc]
        processed = self.location_conv(attn_cat)      # [B, C, T_enc]
        processed = processed.transpose(1, 2)         # [B, T_enc, C]
        processed = self.location_dense(processed)    # [B, T_enc, attn_dim]
        return processed

class LocationSensitiveAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim, location_n_filters=32, location_kernel_size=31):
        super().__init__()
        self.query_layer = nn.Linear(dec_dim, attn_dim, bias=False)
        self.memory_layer = nn.Linear(enc_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=True)
        self.location_layer = LocationLayer(location_n_filters, location_kernel_size, attn_dim)
        self.score_mask_value = -float("inf")

    def get_alignment_energies(self, query, processed_memory, attn_cat):
        # query: [B, dec_dim]
        # processed_memory: [B, T_enc, attn_dim]
        processed_query = self.query_layer(query).unsqueeze(1)    # [B, 1, attn_dim]
        processed_location = self.location_layer(attn_cat)        # [B, T_enc, attn_dim]
        energies = self.v(torch.tanh(processed_query + processed_location + processed_memory)).squeeze(-1)
        return energies                                           # [B, T_enc]

    def forward(self, query, memory, processed_memory, attention_weights_cat, mask):
        alignment = self.get_alignment_energies(query, processed_memory, attention_weights_cat)

        if mask is not None:
            alignment.data.masked_fill_(mask, self.score_mask_value)

        attn_weights = F.softmax(alignment, dim=1)               # [B, T_enc]
        context = torch.bmm(attn_weights.unsqueeze(1), memory).squeeze(1)
        return context, attn_weights

class Decoder(nn.Module):
    def __init__(self, enc_dim, mel_dim, prenet_dim, dec_hidden, attn_dim):
        super().__init__()
        self.mel_dim = mel_dim
        self.prenet = Prenet(mel_dim, sizes=(prenet_dim, prenet_dim), dropout=0.5)

        self.attn_rnn = nn.LSTMCell(prenet_dim + enc_dim, dec_hidden)
        self.attention = LocationSensitiveAttention(
            enc_dim=enc_dim,
            dec_dim=dec_hidden,
            attn_dim=attn_dim,
            location_n_filters=LOCATION_FILTERS,
            location_kernel_size=LOCATION_KERNEL,
        )

        self.decoder_rnn = nn.LSTMCell(dec_hidden + enc_dim, dec_hidden)

        proj_in = dec_hidden + enc_dim
        self.mel_proj = nn.Linear(proj_in, mel_dim * REDUCTION_FACTOR)
        self.stop_proj = nn.Linear(proj_in, 1)

    def initialize_states(self, memory, mask):
        B, T_enc, enc_dim = memory.size()
        self.memory = memory
        self.processed_memory = self.attention.memory_layer(memory)
        self.mask = mask

        self.attn_hidden = memory.new_zeros(B, DEC_HIDDEN)
        self.attn_cell = memory.new_zeros(B, DEC_HIDDEN)
        self.dec_hidden = memory.new_zeros(B, DEC_HIDDEN)
        self.dec_cell = memory.new_zeros(B, DEC_HIDDEN)

        self.attn_weights = memory.new_zeros(B, T_enc)
        self.attn_weights_cum = memory.new_zeros(B, T_enc)
        self.context = memory.new_zeros(B, enc_dim)

    def decode_step(self, prev_mel):
        prenet_out = self.prenet(prev_mel)
        attn_input = torch.cat([prenet_out, self.context], dim=-1)

        self.attn_hidden, self.attn_cell = self.attn_rnn(attn_input, (self.attn_hidden, self.attn_cell))
        self.attn_hidden = F.dropout(self.attn_hidden, 0.1, self.training)

        attn_cat = torch.stack([self.attn_weights, self.attn_weights_cum], dim=1)
        self.context, self.attn_weights = self.attention(
            self.attn_hidden, self.memory, self.processed_memory, attn_cat, self.mask
        )
        self.attn_weights_cum = self.attn_weights_cum + self.attn_weights

        dec_input = torch.cat([self.attn_hidden, self.context], dim=-1)
        self.dec_hidden, self.dec_cell = self.decoder_rnn(dec_input, (self.dec_hidden, self.dec_cell))
        self.dec_hidden = F.dropout(self.dec_hidden, 0.1, self.training)

        proj_input = torch.cat([self.dec_hidden, self.context], dim=-1)
        mel_out = self.mel_proj(proj_input)                       # [B, mel_dim * r]
        stop_logit = self.stop_proj(proj_input).squeeze(-1)      # [B]
        return mel_out, stop_logit, self.attn_weights

class Postnet(nn.Module):
    def __init__(self, mel_dim=80, channels=512, kernel_size=5, num_layers=5):
        super().__init__()
        layers = []
        in_ch = mel_dim
        for i in range(num_layers):
            out_ch = channels if i < num_layers - 1 else mel_dim
            conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=(kernel_size - 1) // 2)
            bn = nn.BatchNorm1d(out_ch)
            layers.append(nn.Sequential(conv, bn))
            in_ch = out_ch
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        # x: [B, T, mel]
        x = x.transpose(1, 2)
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = torch.tanh(x)
            x = F.dropout(x, 0.5, self.training)
        x = x.transpose(1, 2)
        return x

class FromScratchTTS(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim=EMBED_DIM, hidden_dim=ENC_HIDDEN)
        self.decoder = Decoder(
            enc_dim=ENC_HIDDEN * 2,
            mel_dim=N_MELS,
            prenet_dim=PRENET_DIM,
            dec_hidden=DEC_HIDDEN,
            attn_dim=ATTN_DIM,
        )
        self.postnet = Postnet(
            mel_dim=N_MELS,
            channels=POSTNET_CHANNELS,
            kernel_size=POSTNET_KERNEL,
            num_layers=5,
        )

    def forward(self, inputs, input_lens, target_mels=None, teacher_forcing_ratio=1.0,
                max_decoder_steps=DEFAULT_MAX_DECODER_STEPS,
                stop_threshold=DEFAULT_STOP_THRESHOLD,
                min_decoder_steps=DEFAULT_MIN_DECODER_STEPS):
        B = inputs.size(0)
        memory = self.encoder(inputs, input_lens)
        max_enc_len = memory.size(1)

        mask = torch.arange(max_enc_len, device=inputs.device).unsqueeze(0) >= input_lens.unsqueeze(1)
        self.decoder.initialize_states(memory, mask)

        if target_mels is not None:
            target_steps = target_mels.size(1)
            steps = math.ceil(target_steps / REDUCTION_FACTOR)
        else:
            steps = max_decoder_steps

        prev_mel = torch.zeros(B, N_MELS, device=inputs.device)
        mel_outputs = []
        stop_outputs = []
        attn_outputs = []

        for step in range(steps):
            mel_out, stop_logit, attn = self.decoder.decode_step(prev_mel)

            mel_frame = mel_out.view(B, REDUCTION_FACTOR, N_MELS)
            mel_outputs.append(mel_frame)
            stop_outputs.append(stop_logit)
            attn_outputs.append(attn)

            if target_mels is not None and random.random() < teacher_forcing_ratio:
                idx = min((step + 1) * REDUCTION_FACTOR - 1, target_mels.size(1) - 1)
                prev_mel = target_mels[:, idx, :]
            else:
                prev_mel = mel_frame[:, -1, :]

            if target_mels is None and step >= min_decoder_steps:
                if torch.sigmoid(stop_logit).mean().item() > stop_threshold:
                    break

        mel_outputs = torch.cat(mel_outputs, dim=1)               # [B, T, mel]
        stop_outputs = torch.stack(stop_outputs, dim=1)           # [B, T/r]
        attn_outputs = torch.stack(attn_outputs, dim=1)           # [B, T/r, T_enc]

        mel_post = mel_outputs + self.postnet(mel_outputs)
        return mel_outputs, mel_post, stop_outputs, attn_outputs


In [ ]:
# =========================
# 13) Losses and utilities
# =========================
def make_nonpad_mask(lengths, max_len):
    ids = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return ids < lengths.unsqueeze(1)

def masked_l1_loss(pred, target, lengths):
    max_t = min(pred.size(1), target.size(1))
    pred = pred[:, :max_t, :]
    target = target[:, :max_t, :]
    lengths = torch.clamp(lengths, max=max_t)

    mask = make_nonpad_mask(lengths, max_t).unsqueeze(-1).float()
    loss = torch.abs(pred - target) * mask
    return loss.sum() / (mask.sum() * pred.size(-1) + 1e-8)

def masked_stop_bce(stop_logits, stop_targets, mel_lens):
    reduced_lens = torch.ceil(mel_lens.float() / REDUCTION_FACTOR).long()
    T = stop_logits.size(1)
    reduced_lens = torch.clamp(reduced_lens, max=T)

    mask = make_nonpad_mask(reduced_lens, T).float()
    loss = F.binary_cross_entropy_with_logits(
        stop_logits,
        stop_targets[:, :T],
        reduction="none",
        pos_weight=torch.tensor(STOP_POS_WEIGHT, device=stop_logits.device)
    )
    loss = loss * mask
    return loss.sum() / (mask.sum() + 1e-8)

def guided_attention_loss(attn, input_lens, mel_lens, g=GUIDED_ATTN_G):
    # attn: [B, T_dec, T_enc]
    B, T_dec_max, T_enc_max = attn.shape
    total = 0.0
    count = 0

    for b in range(B):
        T_dec = min(attn.size(1), math.ceil(mel_lens[b].item() / REDUCTION_FACTOR))
        T_enc = min(attn.size(2), input_lens[b].item())
        if T_dec <= 1 or T_enc <= 1:
            continue

        t = torch.arange(T_dec, device=attn.device).unsqueeze(1).float() / T_dec
        n = torch.arange(T_enc, device=attn.device).unsqueeze(0).float() / T_enc
        W = 1.0 - torch.exp(-((n - t) ** 2) / (2 * g * g))
        A = attn[b, :T_dec, :T_enc]
        total += torch.mean(A * W)
        count += 1

    if count == 0:
        return torch.tensor(0.0, device=attn.device)
    return total / count

def current_teacher_forcing(epoch, total_epochs):
    if total_epochs <= 1:
        return TEACHER_FORCING_END
    alpha = (epoch - 1) / (total_epochs - 1)
    return TEACHER_FORCING_START + alpha * (TEACHER_FORCING_END - TEACHER_FORCING_START)

def current_guided_lambda(epoch, total_epochs):
    if total_epochs <= 1:
        return GUIDED_ATTN_MIN_LAMBDA
    alpha = (epoch - 1) / (total_epochs - 1)
    return GUIDED_ATTN_MAX_LAMBDA + alpha * (GUIDED_ATTN_MIN_LAMBDA - GUIDED_ATTN_MAX_LAMBDA)

@torch.no_grad()
def plot_attention(attn, save_path):
    plt.figure(figsize=(10, 6))
    plt.imshow(attn, aspect="auto", origin="lower")
    plt.xlabel("Encoder steps")
    plt.ylabel("Decoder steps")
    plt.title("Attention")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

@torch.no_grad()
def plot_mel(mel, save_path, title="Mel Spectrogram"):
    plt.figure(figsize=(10, 4))
    plt.imshow(mel.T, aspect="auto", origin="lower")
    plt.title(title)
    plt.xlabel("Frames")
    plt.ylabel("Mel bins")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


In [ ]:
# =========================
# 14) Build model
# =========================
model = FromScratchTTS(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1, min_lr=5e-5
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", f"{num_params:,}")


In [ ]:
# =========================
# 15) Checkpoint helpers
# =========================
LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"

def save_checkpoint(path, epoch, model, optimizer, scheduler, scaler, history, best_val):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "history": history,
        "best_val": best_val,
        "config": {
            "sample_rate": SAMPLE_RATE,
            "n_fft": N_FFT,
            "win_length": WIN_LENGTH,
            "hop_length": HOP_LENGTH,
            "n_mels": N_MELS,
            "reduction_factor": REDUCTION_FACTOR,
            "vocab": vocab,
        }
    }, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

history = {
    "train_total": [],
    "val_total": [],
    "train_mel": [],
    "val_mel": [],
    "train_stop": [],
    "val_stop": [],
    "train_attn": [],
    "val_attn": [],
    "teacher_forcing": [],
    "guided_lambda": [],
    "lr": [],
}

start_epoch = 1
best_val = float("inf")
epochs_without_improvement = 0

if LATEST_CKPT.exists():
    ckpt = load_checkpoint(LATEST_CKPT, model, optimizer, scheduler, scaler)
    start_epoch = ckpt["epoch"] + 1
    history = ckpt.get("history", history)
    best_val = ckpt.get("best_val", best_val)
    print(f"Resumed from epoch {ckpt['epoch']}, best val = {best_val:.4f}")
else:
    print("No previous checkpoint found. Starting fresh.")


In [ ]:
# =========================
# 16) Inference helpers
# =========================
def token_ids_from_text(text):
    text = clean_text(text)
    tok_str = text_to_token_string(text)
    toks = tok_str.split()
    ids = [SOS_ID] + [stoi[t] for t in toks if t in stoi] + [EOS_ID]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0), tok_str

@torch.no_grad()
def synthesize_mel(model, text,
                   max_steps=DEFAULT_MAX_DECODER_STEPS,
                   stop_threshold=DEFAULT_STOP_THRESHOLD,
                   min_decoder_steps=DEFAULT_MIN_DECODER_STEPS):
    model.eval()

    ids, tok_str = token_ids_from_text(text)
    ids = ids.to(device)
    lengths = torch.tensor([ids.size(1)], dtype=torch.long, device=device)

    with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
        mel_out, mel_post, stop_logits, attn = model(
            ids,
            lengths,
            target_mels=None,
            teacher_forcing_ratio=0.0,
            max_decoder_steps=max_steps,
            stop_threshold=stop_threshold,
            min_decoder_steps=min_decoder_steps,
        )

    mel = mel_post.squeeze(0).detach().cpu().float().numpy()
    stop_probs = torch.sigmoid(stop_logits).squeeze(0).detach().cpu().float().numpy()
    attn = attn.squeeze(0).detach().cpu().float().numpy()

    return {
        "tokens": tok_str,
        "mel": mel,
        "stop_probs": stop_probs,
        "attn": attn,
    }

@torch.no_grad()
def text_to_audio_file(model, text, out_wav_path,
                       stop_threshold=DEFAULT_STOP_THRESHOLD,
                       griffin_iters=80):
    result = synthesize_mel(
        model,
        text,
        stop_threshold=stop_threshold,
        min_decoder_steps=DEFAULT_MIN_DECODER_STEPS,
        max_steps=DEFAULT_MAX_DECODER_STEPS,
    )
    audio = log_mel_to_audio_griffinlim(result["mel"], n_iter=griffin_iters)
    sf.write(out_wav_path, audio, SAMPLE_RATE)

    plot_mel(result["mel"], str(Path(out_wav_path).with_suffix(".png")), title="Predicted Mel")
    plot_attention(result["attn"], str(Path(out_wav_path).with_name(Path(out_wav_path).stem + "_attn.png")))

    return result


In [ ]:
# =========================
# 17) Training + validation loop
# =========================
@torch.no_grad()
def validate(model, loader, epoch):
    model.eval()

    total_loss_sum = 0.0
    mel_loss_sum = 0.0
    stop_loss_sum = 0.0
    attn_loss_sum = 0.0
    total_items = 0

    for batch in loader:
        ids, input_lens, mels, mel_lens, stop_targets, _, _ = batch
        ids = ids.to(device, non_blocking=True)
        input_lens = input_lens.to(device, non_blocking=True)
        mels = mels.to(device, non_blocking=True)
        mel_lens = mel_lens.to(device, non_blocking=True)
        stop_targets = stop_targets.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            mel_pred, mel_post, stop_logits, attn = model(
                ids,
                input_lens,
                target_mels=mels,
                teacher_forcing_ratio=1.0,
            )
            mel_loss = masked_l1_loss(mel_pred, mels, mel_lens) + masked_l1_loss(mel_post, mels, mel_lens)
            stop_loss = masked_stop_bce(stop_logits, stop_targets, mel_lens)
            attn_loss = guided_attention_loss(attn, input_lens, mel_lens)
            attn_lambda = current_guided_lambda(epoch, EPOCHS)
            total_loss = mel_loss + stop_loss + attn_lambda * attn_loss

        bs = ids.size(0)
        total_loss_sum += total_loss.item() * bs
        mel_loss_sum += mel_loss.item() * bs
        stop_loss_sum += stop_loss.item() * bs
        attn_loss_sum += attn_loss.item() * bs
        total_items += bs

    return (
        total_loss_sum / max(total_items, 1),
        mel_loss_sum / max(total_items, 1),
        stop_loss_sum / max(total_items, 1),
        attn_loss_sum / max(total_items, 1),
    )

def train_one_epoch(model, loader, epoch):
    model.train()

    teacher_forcing = current_teacher_forcing(epoch, EPOCHS)
    attn_lambda = current_guided_lambda(epoch, EPOCHS)

    total_loss_sum = 0.0
    mel_loss_sum = 0.0
    stop_loss_sum = 0.0
    attn_loss_sum = 0.0
    total_items = 0

    for batch in loader:
        ids, input_lens, mels, mel_lens, stop_targets, _, _ = batch
        ids = ids.to(device, non_blocking=True)
        input_lens = input_lens.to(device, non_blocking=True)
        mels = mels.to(device, non_blocking=True)
        mel_lens = mel_lens.to(device, non_blocking=True)
        stop_targets = stop_targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            mel_pred, mel_post, stop_logits, attn = model(
                ids,
                input_lens,
                target_mels=mels,
                teacher_forcing_ratio=teacher_forcing,
            )

            mel_loss = masked_l1_loss(mel_pred, mels, mel_lens) + masked_l1_loss(mel_post, mels, mel_lens)
            stop_loss = masked_stop_bce(stop_logits, stop_targets, mel_lens)
            attn_loss = guided_attention_loss(attn, input_lens, mel_lens)
            total_loss = mel_loss + stop_loss + attn_lambda * attn_loss

        if not torch.isfinite(total_loss):
            print("Skipping non-finite batch.")
            continue

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        bs = ids.size(0)
        total_loss_sum += total_loss.item() * bs
        mel_loss_sum += mel_loss.item() * bs
        stop_loss_sum += stop_loss.item() * bs
        attn_loss_sum += attn_loss.item() * bs
        total_items += bs

    return (
        total_loss_sum / max(total_items, 1),
        mel_loss_sum / max(total_items, 1),
        stop_loss_sum / max(total_items, 1),
        attn_loss_sum / max(total_items, 1),
        teacher_forcing,
        attn_lambda,
    )


In [ ]:
# =========================
# 18) Sample generation after each epoch
# =========================
SAMPLE_TEXT = "Hello, this is my graduation project speaking from a model built from scratch."

@torch.no_grad()
def save_epoch_sample(model, epoch, label="latest"):
    model.eval()
    out_wav = SAMPLE_DIR / f"epoch_{epoch:02d}_{label}.wav"
    result = text_to_audio_file(model, SAMPLE_TEXT, str(out_wav), stop_threshold=DEFAULT_STOP_THRESHOLD, griffin_iters=80)
    print("Saved sample:", out_wav)
    print("Tokens:", result["tokens"])


In [ ]:
# =========================
# 19) Run training
# =========================
if start_epoch > EPOCHS:
    print("Training already reached configured EPOCHS. Skipping training loop.")
else:
    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()

        train_total, train_mel, train_stop, train_attn, tf_ratio, attn_lambda = train_one_epoch(model, train_loader, epoch)
        val_total, val_mel, val_stop, val_attn = validate(model, val_loader, epoch)

        scheduler.step(val_total)

        history["train_total"].append(train_total)
        history["val_total"].append(val_total)
        history["train_mel"].append(train_mel)
        history["val_mel"].append(val_mel)
        history["train_stop"].append(train_stop)
        history["val_stop"].append(val_stop)
        history["train_attn"].append(train_attn)
        history["val_attn"].append(val_attn)
        history["teacher_forcing"].append(tf_ratio)
        history["guided_lambda"].append(attn_lambda)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        save_checkpoint(LATEST_CKPT, epoch, model, optimizer, scheduler, scaler, history, best_val)

        improved = val_total < best_val
        if improved:
            best_val = val_total
            epochs_without_improvement = 0
            save_checkpoint(BEST_CKPT, epoch, model, optimizer, scheduler, scaler, history, best_val)
            save_epoch_sample(model, epoch, label="best")
            tag = "Saved best checkpoint."
        else:
            epochs_without_improvement += 1
            save_epoch_sample(model, epoch, label="latest")
            tag = f"No improvement ({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})."

        mins = (time.time() - epoch_start) / 60.0
        print(
            f"Epoch {epoch:02d} | "
            f"Train Total {train_total:.4f} | Val Total {val_total:.4f} | "
            f"Train Mel {train_mel:.4f} | Val Mel {val_mel:.4f} | "
            f"Train Stop {train_stop:.4f} | Val Stop {val_stop:.4f} | "
            f"Train Attn {train_attn:.4f} | Val Attn {val_attn:.4f} | "
            f"TF {tf_ratio:.3f} | Attn λ {attn_lambda:.3f} | "
            f"LR {optimizer.param_groups[0]['lr']:.6f} | "
            f"{mins:.1f} min | {tag}"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break


In [ ]:
# =========================
# 20) Load best checkpoint for final inference
# =========================
if BEST_CKPT.exists():
    _ = load_checkpoint(BEST_CKPT, model)
    print("Loaded best checkpoint:", BEST_CKPT)
else:
    print("Best checkpoint not found. Using current model weights.")


In [ ]:
# =========================
# 21) Training curves
# =========================
if len(history["train_total"]) > 0:
    epochs_axis = np.arange(1, len(history["train_total"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_axis, history["train_total"], label="Train Total")
    plt.plot(epochs_axis, history["val_total"], label="Val Total")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Total Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "loss_total.png")
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_axis, history["train_mel"], label="Train Mel")
    plt.plot(epochs_axis, history["val_mel"], label="Val Mel")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Mel Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "loss_mel.png")
    plt.show()


## Final inference
Run the next cell with any text you want.  
It will save:
- predicted waveform `.wav`
- predicted mel plot `.png`
- attention plot `.png`


In [ ]:
# =========================
# 22) Final text -> audio inference
# =========================
TEST_TEXT = "Hello, this is my final from scratch text to speech model for my graduation project."

final_wav_path = WORK_DIR / "final_demo.wav"
result = text_to_audio_file(
    model,
    TEST_TEXT,
    str(final_wav_path),
    stop_threshold=DEFAULT_STOP_THRESHOLD,
    griffin_iters=80,
)

print("Saved final wav to:", final_wav_path)
print("Tokens:", result["tokens"])
print("Mel frames:", result["mel"].shape[0])
print("Audio duration (sec):", len(sf.read(final_wav_path)[0]) / SAMPLE_RATE)


In [ ]:
# =========================
# 23) Listen in Kaggle
# =========================
from IPython.display import Audio, display
display(Audio(str(final_wav_path), rate=SAMPLE_RATE))


## What to hand in
For your submission, the key artifacts are usually:

- `best.pt`
- `final_demo.wav`
- the notebook itself
- a short explanation that the model is **from scratch**
- a note that waveform generation uses **Griffin-Lim** as a fully from-scratch vocoder baseline

## Why this still counts as from scratch
- no pretrained acoustic model
- no pretrained vocoder
- random initialization
- trained only on your own attached dataset
- inference pipeline implemented in notebook
